# Multiperiod HEN Synthesis

**Learning outcome:** Apply multiperiod hen synthesis through the public `PinchProblem` or `PinchWorkspace` workflow.

**Level:** Advanced  
**Execution profile:** `solver`  
**Expected runtime:** 10 to 90 minutes  
**Optional extras:** hen

## Study question and data

**Study question:** Can one exchanger network serve every operating period without hiding period constraints?

## Step 1: Target periods and synthesize a shared network

This cell builds a two-period case, with `base` and `turndown` weighted 0.7 and 0.3. `all_periods.all_heat_integration()` targets every period first. `multiperiod_heat_exchanger_network(stages=[3], best_solutions=1)` then synthesises one shared network. `ranked_networks` is its best successful outcome.

In [ ]:
from OpenPinch import PinchProblem

base_problem = PinchProblem(
    "Four-stream-Yee-and-Grossmann-1990-1.json"
)
multiperiod_input = base_problem.to_problem_json()
multiperiod_input["options"]["PROBLEM_PERIOD_IDS"] = [
    "base", "turndown"
]
multiperiod_input["options"]["PROBLEM_PERIOD_WEIGHTS"] = [
    0.7, 0.3
]
for stream in multiperiod_input["streams"]:
    duty = stream["heat_flow"]
    stream["heat_flow"] = {
        "values": [duty["value"], 0.8 * duty["value"]],
        "unit": duty["unit"],
    }
    stream["heat_capacity_flowrate"] = None
problem = PinchProblem(
    multiperiod_input, project_name="Four Stream Periods"
)
period_targets = problem.target.all_periods.all_heat_integration()
design = problem.design.multiperiod_heat_exchanger_network(
    stages=[3], best_solutions=1
)
ranked_networks = design.top(1)

## Step 2: Inspect the period-specific shared design

This cell selects the top-ranked shared network as `shared_network`. `grid(rank=1, period_id=...)` renders it for the first period as `shared_grid`. The final line shows the network's `summary_metrics`.

In [ ]:
shared_network = design.network(rank=1)
period_ids = list(problem.period_results)
shared_grid = design.grid(rank=1, period_id=period_ids[0])
shared_network.summary_metrics

## Review the result

Review the shared network metrics and one period-specific grid with the ranking to identify the period that controls the common design.

In [ ]:
from IPython.display import display

display(ranked_networks)
display(shared_network.summary_metrics)
display(shared_grid)

## Interpret the result

Inspect the shared network grid in each period and identify the period controlling area and utility demand.

## Adapt this template

Use representative periods and verified weights; increase stages only after a smaller shared-design model is feasible.